In [14]:
import uuid
import os
import re
import io
import argparse
import torch
import numpy as np
from PIL import Image

from tool_server.utils.utils import *
# from tool_server.utils.server_utils import *
import matplotlib.pyplot as plt

# from tool_server.tool_workers.online_workers.base_tool_worker import BaseToolWorker

from transformers import AutoModelForCausalLM, AutoProcessor, GenerationConfig, BitsAndBytesConfig

In [15]:
# load the processor
model_path = "/mnt/petrelfs/songmingyang/songmingyang/model/mm/tools/Molmo-7B-D-0924"
processor = AutoProcessor.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype='auto',
    device_map='auto'
)

# load the model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype='auto',
    device_map='auto'
)



Loading checkpoint shards: 100%|██████████| 7/7 [00:09<00:00,  1.35s/it]


In [21]:

text_prompt = "Point to the {} in the scene.".format("tire")
text_prompt2 = "Point to the {} in the scene.".format("Nose")
text_prompt3 = "Point to the {} in the scene.".format("Zebra")

zebra_path = "/mnt/petrelfs/songmingyang/code/reasoning/tool-agent/tool_server/tool_workers/online_workers/test_cases/worker_tests/input_cases/zebra.jpg"
truck_path ="/mnt/petrelfs/songmingyang/code/reasoning/tool-agent/tool_server/tool_workers/online_workers/test_cases/worker_tests/input_cases/truck.jpg"

zebra = Image.open(zebra_path).convert("RGB")
truck = Image.open(truck_path).convert("RGB")
images = [zebra, zebra, zebra]
texts = [text_prompt,text_prompt2,text_prompt3]



tokenizer = processor.tokenizer
tok_inputs = tokenizer(texts)
input_ids,attention_mask = tok_inputs["input_ids"],tok_inputs["attention_mask"]

inputs = processor.process(
    images=images,
    text=text_prompt,
)
inputs["images"] = inputs["images"].to(torch.bfloat16)
inputs = {k: v.to(model.device).unsqueeze(0) for k, v in inputs.items()}

In [23]:
# 为每个(图像,文本)对单独处理输入
processed_inputs = []
for img, txt in zip(images, texts):
    inputs = processor.process(images=[img], text=txt)
    # 不需要在这里移动到设备
    processed_inputs.append(inputs)

# 让我们看一下各个输入的形状，以便正确创建批量张量
print("检查各个处理后输入的形状:")
for i, inputs in enumerate(processed_inputs):
    print(f"Sample {i}:")
    for k, v in inputs.items():
        print(f"  {k}: {v.shape}")

# 对每个输入分别移动到正确的设备上
batch = {}
batch["input_ids"] = torch.tensor(tok_inputs["input_ids"])
batch["attention_mask"] = torch.tensor(tok_inputs["attention_mask"])
for key in processed_inputs[0].keys():
   
    
    if key == "images":
        # 对于图像，如果尺寸不同，这里可能需要额外的处理
        # 但Molmo的processor应该已经将所有图像处理为相同的维度了
        try:
            tensors = [inp[key].unsqueeze(0) for inp in processed_inputs]
            batch[key] = torch.cat(tensors, dim=0).to(model.device)
        except RuntimeError as e:
            print(f"图像尺寸不匹配: {e}")
            print("各图像的形状:")
            for i, inp in enumerate(processed_inputs):
                print(f"  Sample {i}: {inp[key].shape}")
            
            # 由于图像尺寸不匹配，我们可以改为单独处理每个样本
            print("将改为单独处理每个样本")
            batch = None
            break
    
    elif key == "image_masks" or key == "image_input_idx":
        # 对于其他张量也需要确保尺寸匹配
        try:
            tensors = [inp[key].unsqueeze(0) for inp in processed_inputs]
            batch[key] = torch.cat(tensors, dim=0).to(model.device)
        except RuntimeError as e:
            print(f"{key}尺寸不匹配: {e}")
            batch = None
            break

# 如果成功创建批次，则批量推理
if batch is not None:
    print("执行批量推理...")
    outputs = model.generate_from_batch(
        batch,
        GenerationConfig(max_new_tokens=200, stop_strings="<|endoftext|>"),
        tokenizer=processor.tokenizer
    )
    
    # 处理生成的输出
    results = []
    for i, output in enumerate(outputs):
        input_length = processed_inputs[i]['input_ids'].size(0)
        generated_tokens = output[input_length:]
        generated_text = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        results.append(generated_text)
    
    # 打印生成的文本
    for i, result in enumerate(results):
        print(f"Query {i+1}: {texts[i]}")
        print(f"Response: {result}")
        print("-" * 50)
else:
    # 如果批处理不成功，退回到逐个样本处理
    print("执行单样本推理...")
    results = []
    
    for i, (img, txt) in enumerate(zip(images, texts)):
        print(f"处理样本 {i+1}...")
        # 重新处理输入，并直接移到设备上
        inputs = processor.process(images=[img], text=txt)
        inputs = {k: v.to(model.device).unsqueeze(0) for k, v in inputs.items()}
        
        # 生成输出
        output = model.generate_from_batch(
            inputs,
            GenerationConfig(max_new_tokens=200, stop_strings="<|endoftext|>"),
            tokenizer=processor.tokenizer
        )
        
        # 提取生成的文本
        generated_tokens = output[0, inputs['input_ids'].size(1):]
        generated_text = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)
        results.append(generated_text)
    
    # 打印结果
    for i, result in enumerate(results):
        print(f"Query {i+1}: {texts[i]}")
        print(f"Response: {result}")
        print("-" * 50)

检查各个处理后输入的形状:
Sample 0:
  input_ids: torch.Size([425])
  images: torch.Size([3, 576, 588])
  image_input_idx: torch.Size([3, 144])
  image_masks: torch.Size([3, 576])
Sample 1:
  input_ids: torch.Size([425])
  images: torch.Size([3, 576, 588])
  image_input_idx: torch.Size([3, 144])
  image_masks: torch.Size([3, 576])
Sample 2:
  input_ids: torch.Size([426])
  images: torch.Size([3, 576, 588])
  image_input_idx: torch.Size([3, 144])
  image_masks: torch.Size([3, 576])
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/mnt/petrelfs/songmingyang/anaconda3/envs/tool_server/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3508, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_217990/3704192439.py", line 17, in <module>
    batch["input_ids"] = torch.tensor(tok_inputs["input_ids"])
ValueError: expected sequence of length 8 at dim 1 (got 9)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/mnt/petrelfs/songmingyang/anaconda3/envs/tool_server/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 2105, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
  File "/mnt/petrelfs/songmingyang/anaconda3/envs/tool_server/lib/python3.10/site-packages/IPython/core/ultratb.py", line 1396, in structured_traceback
    return FormattedTB.structured_traceback(
  File "/mnt/petrelfs/songmingyang/anaconda3/envs/tool_server/lib/

In [14]:
input_ids["input_ids"]

[[2609, 311, 279, 7610, 304, 279, 6109, 13],
 [2609, 311, 279, 92123, 304, 279, 6109, 13],
 [2609, 311, 279, 1863, 50213, 304, 279, 6109, 13]]

In [6]:

with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=torch.bfloat16):

        output = model.generate_from_batch(
            inputs,
            GenerationConfig(max_new_tokens=1024, stop_strings="<|endoftext|>"),
            tokenizer=processor.tokenizer
        )

        # only get generated tokens; decode them to text
        generated_tokens = output[0,inputs['input_ids'].size(1):]
        response = processor.tokenizer.decode(generated_tokens, skip_special_tokens=True)


In [7]:
response

' <point x="5.5" y="17.9" alt="Point E">Point E</point>'